# Study Planner — Dataset Preprocessing & Cleaning

**Component:** Academic Management with Personalized Study Planner (Smart Uni Guide)

This notebook turns the raw **OULAD** (Open University Learning Analytics Dataset) files into one clean, **task-level, leakage-free** table that the priority model is trained on:

`ml_scripts/study-planner/outputs/oulad_task_level_leakage_free.csv` — one row per (student, assessment), 13 features + `Priority_Label`.

**Pipeline**

1. Load the raw OULAD tables
2. Data-quality check (`?` placeholders, missing values)
3. Clean (drop rows that can't be used)
4. Join the tables into one task-level table
5. Engineer engagement features from the 10M-row click log (chunked)
6. Engineer a leakage-free `prior_avg_score`
7. Build the outcome-based target `Priority_Label`
8. Drop identifier / leakage / fairness-sensitive columns
9. Encode categoricals and export
10. Exploratory data analysis on the final table

Run the next notebook, `model_training.ipynb`, after this one.

## 1. Setup

In [1]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


In [2]:
from pathlib import Path


def find_backend_dir(start=None):
    """Walks up from the notebook's folder to backend/ (the folder holding app/ and ml_scripts/)."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "app").is_dir() and (p / "ml_scripts").is_dir():
            return p
    raise FileNotFoundError("Open this notebook from inside the backend/ folder tree.")


BACKEND_DIR = find_backend_dir()
DATASETS_DIR = BACKEND_DIR / "datasets" / "study-planner"
OUTPUTS_DIR = BACKEND_DIR / "ml_scripts" / "study-planner" / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_CSV_PATH = OUTPUTS_DIR / "oulad_task_level_leakage_free.csv"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42


def save_fig(name):
    """Charts are saved as PNG files (shown in the markdown cell below each one) rather than
    embedded in the notebook, which keeps the .ipynb small enough to diff and review."""
    path = FIGURES_DIR / f"{name}.png"
    plt.savefig(path, dpi=110, bbox_inches="tight")
    plt.close("all")
    print("Saved figure:", path.relative_to(BACKEND_DIR))


print("Backend dir :", BACKEND_DIR)
print("Datasets dir:", DATASETS_DIR)

Backend dir : C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend
Datasets dir: C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend\datasets\study-planner


## 2. Load the raw OULAD tables

| File | Used for |
|---|---|
| `assessments.csv` | assessment type, **deadline (`date`)**, `weight` |
| `studentAssessment.csv` | per-student score and submission date (used **only** for the label) |
| `studentInfo.csv` | previous attempts, studied credits |
| `studentRegistration.csv` | registration date |
| `courses.csv` | module presentation length |
| `studentVle.csv` | daily click logs (~10.6M rows) → engagement features (section 6) |

In [3]:
assessments = pd.read_csv(DATASETS_DIR / "assessments.csv")
student_assessment = pd.read_csv(DATASETS_DIR / "studentAssessment.csv")
student_info = pd.read_csv(DATASETS_DIR / "studentInfo.csv")
student_registration = pd.read_csv(DATASETS_DIR / "studentRegistration.csv")
courses = pd.read_csv(DATASETS_DIR / "courses.csv")

raw_tables = {
    "assessments": assessments,
    "studentAssessment": student_assessment,
    "studentInfo": student_info,
    "studentRegistration": student_registration,
    "courses": courses,
}
pd.DataFrame({name: {"rows": t.shape[0], "columns": t.shape[1]} for name, t in raw_tables.items()}).T

,rows,columns
assessments,206,6
studentAssessment,173912,5
studentInfo,32593,12
studentRegistration,32593,5
courses,22,3


In [4]:
assessments.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19,10.0
1,AAA,2013J,1753,TMA,54,20.0
2,AAA,2013J,1754,TMA,117,20.0
3,AAA,2013J,1755,TMA,166,20.0
4,AAA,2013J,1756,TMA,215,30.0


In [5]:
student_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79


## 3. Data-quality check

OULAD marks unknown values with a literal `?` string instead of leaving the cell empty, so both real `NaN`s **and** `?` placeholders are counted.

In [6]:
rows = []
for name, table in raw_tables.items():
    for col in table.columns:
        n_nan = int(table[col].isna().sum())
        n_q = int((table[col].astype(str) == "?").sum())
        if n_nan or n_q:
            rows.append({"table": name, "column": col, "NaN": n_nan, "'?' placeholders": n_q,
                         "% of rows": round(100 * (n_nan + n_q) / len(table), 2)})
pd.DataFrame(rows)

,table,column,NaN,'?' placeholders,% of rows
0,assessments,date,0,11,5.34
1,studentAssessment,score,0,173,0.10
2,studentInfo,imd_band,0,1111,3.41
3,studentRegistration,date_registration,0,45,0.14
4,studentRegistration,date_unregistration,0,22521,69.10


## 4. Cleaning

* **`assessments.date == '?'`** — these are the final *Exam* rows, whose date is set at module level, not per assessment. The model predicts per-assessment planning priority, so they are dropped.
* **`studentAssessment.score == '?'`** — an attempt with no score can't produce an outcome-based label, so it is dropped.
* **`date_registration`** — `?` is coerced to `NaN` (imputed later, in training).

In [7]:
before = len(assessments)
assessments = assessments[assessments["date"] != "?"].copy()
assessments["date"] = assessments["date"].astype(int)
print(f"assessments: dropped {before - len(assessments)} rows with date == '?' (Exam-type rows)")

before = len(student_assessment)
student_assessment = student_assessment[student_assessment["score"] != "?"].copy()
student_assessment["score"] = student_assessment["score"].astype(float)
print(f"studentAssessment: dropped {before - len(student_assessment)} rows with score == '?'")

student_registration = student_registration.copy()
student_registration["date_registration"] = pd.to_numeric(student_registration["date_registration"], errors="coerce")
print(f"studentRegistration: {student_registration['date_registration'].isna().sum()} date_registration values are NaN")

assessments: dropped 11 rows with date == '?' (Exam-type rows)
studentAssessment: dropped 173 rows with score == '?'
studentRegistration: 45 date_registration values are NaN


## 5. Join into one task-level table

`studentAssessment ⋈ assessments` is an **inner** join (only assessments that were actually attempted and scored). Everything else is a **left** join so no task row is lost.

In [8]:
shapes = []
df = student_assessment.merge(assessments, on="id_assessment", how="inner")
shapes.append(("studentAssessment + assessments (inner)", df.shape))

df = df.merge(
    student_info[["code_module", "code_presentation", "id_student", "num_of_prev_attempts", "studied_credits",
                  "disability", "highest_education", "age_band", "final_result"]],
    on=["code_module", "code_presentation", "id_student"], how="left",
)
shapes.append(("+ studentInfo (left)", df.shape))

df = df.merge(
    student_registration[["code_module", "code_presentation", "id_student", "date_registration"]],
    on=["code_module", "code_presentation", "id_student"], how="left",
)
shapes.append(("+ studentRegistration (left)", df.shape))

df = df.merge(
    courses[["code_module", "code_presentation", "module_presentation_length"]],
    on=["code_module", "code_presentation"], how="left",
)
shapes.append(("+ courses (left)", df.shape))
pd.DataFrame(shapes, columns=["step", "shape"])

,step,shape
0,studentAssessment + assessments (inner),"(170874, 10)"
1,+ studentInfo (left),"(170874, 16)"
2,+ studentRegistration (left),"(170874, 17)"
3,+ courses (left),"(170874, 18)"


## 6. Engagement features from `studentVle.csv` (chunked)

The click log is ~423 MB / 10.6M rows, so it is streamed in 500k-row chunks and only a **weekly** summary is kept in memory. Per student-module:

| Feature | Meaning |
|---|---|
| `avg_weekly_clicks` | mean clicks per week |
| `clicks_trend` | last week's clicks − mean of all earlier weeks (rising / fading effort) |
| `active_weeks_ratio` | share of weeks with any activity |
| `has_vle_activity` | 1 if the student has any click record at all, else 0 |

*This cell takes a few minutes.*

In [9]:
VLE_PATH = DATASETS_DIR / "studentVle.csv"
GROUP_KEYS = ["code_module", "code_presentation", "id_student"]

t0 = time.time()
weekly_parts, n_rows = [], 0
for i, chunk in enumerate(pd.read_csv(VLE_PATH, chunksize=500_000)):
    chunk["week"] = chunk["date"] // 7  # week relative to module start (negative = before start)
    weekly_parts.append(chunk.groupby(GROUP_KEYS + ["week"])["sum_click"].sum().reset_index())
    n_rows += len(chunk)

# A student/week can straddle two chunks, so sum the partial aggregates again.
weekly_clicks = (
    pd.concat(weekly_parts, ignore_index=True)
    .groupby(GROUP_KEYS + ["week"])["sum_click"].sum().reset_index()
)
print(f"Read {n_rows:,} raw click rows in {time.time() - t0:.1f}s -> weekly table {weekly_clicks.shape}")

Read 10,655,280 raw click rows in 6.4s -> weekly table (627031, 5)


In [10]:
def summarize_group(g):
    g = g.sort_values("week")
    if len(g) >= 2:
        clicks_trend = g["sum_click"].iloc[-1] - g["sum_click"].iloc[:-1].mean()
    else:
        clicks_trend = 0.0
    return pd.Series({
        "avg_weekly_clicks": g["sum_click"].mean(),
        "clicks_trend": clicks_trend,
        "active_weeks_ratio": (g["sum_click"] > 0).mean(),
    })


t0 = time.time()
vle_features = weekly_clicks.groupby(GROUP_KEYS).apply(summarize_group).reset_index()
vle_features["has_vle_activity"] = True
print(f"Engagement features for {len(vle_features):,} student-modules in {time.time() - t0:.1f}s")
vle_features.describe().round(2)

Engagement features for 29,228 student-modules in 8.8s


,id_student,avg_weekly_clicks,clicks_trend,active_weeks_ratio
count,29228.00,29228.00,29228.00,29228.0
mean,708336.68,51.40,-25.57,1.0
std,551736.16,46.35,89.47,0.0
min,6516.00,1.00,-630.82,1.0
25%,507576.75,21.69,-53.77,1.0
50%,589828.00,37.11,-24.29,1.0
75%,643706.75,66.39,-4.37,1.0
max,2698588.00,652.68,2713.80,1.0


In [11]:
df = df.merge(vle_features, on=GROUP_KEYS, how="left")

# Students with no click record at all: engagement = 0 and flagged via has_vle_activity.
engagement_cols = ["avg_weekly_clicks", "clicks_trend", "active_weeks_ratio"]
df["has_vle_activity"] = df["has_vle_activity"].fillna(False)
df[engagement_cols] = df[engagement_cols].fillna(0.0)
print(f"Task rows with no VLE activity: {(~df['has_vle_activity'].astype(bool)).sum():,} of {len(df):,}")

Task rows with no VLE activity: 166 of 170,874


## 7. Leakage-free `prior_avg_score`

The student's average score on **earlier-deadline** assessments of the same module — the current assessment's own score is excluded (`shift(1)`), otherwise the model would "see the future".

For a student's first assessment (no history yet) the fallback is the module-presentation cohort mean, then the overall dataset mean.

In [12]:
df = df.sort_values(["id_student", "code_module", "code_presentation", "date"]).reset_index(drop=True)

grp = df.groupby(["id_student", "code_module", "code_presentation"])["score"]
prior = grp.transform(lambda s: s.expanding().mean().shift(1))
n_first = int(prior.isna().sum())

cohort_mean = df.groupby(["code_module", "code_presentation"])["score"].transform("mean")
overall_mean = df["score"].mean()
df["prior_avg_score"] = prior.fillna(cohort_mean).fillna(overall_mean)

print(f"Rows with no earlier assessment (cohort/overall-mean fallback): {n_first:,}")
print(f"Overall dataset mean score (cold-start default used by the app): {overall_mean:.6f}")

Rows with no earlier assessment (cohort/overall-mean fallback): 25,819
Overall dataset mean score (cold-start default used by the app): 75.946212


## 8. Target label — `Priority_Label` (outcome-based)

> **Data-leakage lesson.** The first version built the label from `weight`, `date` and `prior_avg_score` — the same columns used as inputs — so every model scored 99%+ by simply re-learning the formula. A suspiciously perfect score is a red flag, not a success.

The label now comes from what **actually happened** on the task (known only in hindsight):

* `submitted_late = date_submitted > date`
* **High** — `score < 50` **or** submitted late **or** (`score < 60` **and** `weight ≥ 20`)
* **Low** — `score ≥ 75` **and** on time **and** `weight < 15`
* **Medium** — everything else

`score`, `date_submitted` and `submitted_late` are used **only here** and dropped in section 9, so they never become model inputs.

In [13]:
df["submitted_late"] = (df["date_submitted"] > df["date"]).astype(int)


def label_from_outcome(row):
    if row["score"] < 50 or row["submitted_late"] == 1 or (row["score"] < 60 and row["weight"] >= 20):
        return "High"
    if row["score"] >= 75 and row["submitted_late"] == 0 and row["weight"] < 15:
        return "Low"
    return "Medium"


df["Priority_Label"] = df.apply(label_from_outcome, axis=1)

dist = pd.DataFrame({
    "count": df["Priority_Label"].value_counts(),
    "share": df["Priority_Label"].value_counts(normalize=True).round(3),
}).loc[["High", "Medium", "Low"]]
dist

,count,share
Priority_Label,,
High,60518,0.354
Medium,62984,0.369
Low,47372,0.277


In [14]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = {"High": "#e5484d", "Medium": "#f5a524", "Low": "#30a46c"}
ax.bar(dist.index, dist["count"], color=[colors[c] for c in dist.index])
for i, (c, s) in enumerate(zip(dist["count"], dist["share"])):
    ax.text(i, c, f"{s:.1%}", ha="center", va="bottom")
ax.set_title("Priority_Label distribution (no resampling needed)")
ax.set_ylabel("Task rows")
save_fig("prep_label_distribution")

Saved figure: ml_scripts\study-planner\outputs\figures\prep_label_distribution.png


![Priority_Label distribution](outputs/figures/prep_label_distribution.png)

## 9. Drop identifier, leakage and fairness-sensitive columns

| Dropped | Reason |
|---|---|
| `id_assessment`, `id_student` | identifiers |
| `score`, `date_submitted`, `is_banked`, `submitted_late` | **leakage** — only known after the task is done (used for the label only) |
| `final_result` | **leakage** — only known at the end of the module |
| `disability`, `highest_education`, `age_band` | fairness-sensitive — deliberately not used for prioritising a student's work |

In [15]:
drop_cols = [
    "id_assessment", "id_student",
    "score", "date_submitted", "is_banked",
    "days_early_or_late", "submitted_late",
    "final_result",
    "disability", "highest_education", "age_band",
]
present_drop_cols = [c for c in drop_cols if c in df.columns]
df_model = df.drop(columns=present_drop_cols)
print("Dropped:", present_drop_cols)

feature_cols = [
    "date", "weight", "assessment_type",
    "num_of_prev_attempts", "studied_credits", "module_presentation_length",
    "date_registration", "prior_avg_score",
    "code_module",
    "avg_weekly_clicks", "clicks_trend", "active_weeks_ratio", "has_vle_activity",
]
leaky = {"score", "date_submitted", "submitted_late", "final_result"} & set(feature_cols)
assert not leaky, f"Leakage check failed: {leaky}"
print("Leakage check passed - no outcome column is a feature.")

Dropped: ['id_assessment', 'id_student', 'score', 'date_submitted', 'is_banked', 'submitted_late', 'final_result', 'disability', 'highest_education', 'age_band']
Leakage check passed - no outcome column is a feature.


## 10. Encode categoricals and export

`assessment_type` and `code_module` are label-encoded (alphabetical, so the mapping is deterministic and matches the deployed `label_encoders.joblib`).

The final column order puts **`date` at index 0**: the training notebook's deadline constraint relies on that.

In [16]:
le_assessment_type = LabelEncoder()
df_model["assessment_type_enc"] = le_assessment_type.fit_transform(df_model["assessment_type"].astype(str))
le_code_module = LabelEncoder()
df_model["code_module_enc"] = le_code_module.fit_transform(df_model["code_module"].astype(str))
df_model["has_vle_activity"] = df_model["has_vle_activity"].astype(int)

for name, le in [("assessment_type", le_assessment_type), ("code_module", le_code_module)]:
    print(f"{name}: {dict(zip(le.classes_, range(len(le.classes_))))}")

model_feature_cols = [c for c in feature_cols if c not in ("assessment_type", "code_module")]
model_feature_cols += ["assessment_type_enc", "code_module_enc"]
print(f"\n{len(model_feature_cols)} model features: {model_feature_cols}")

assessment_type: {'CMA': 0, 'Exam': 1, 'TMA': 2}
code_module: {'AAA': 0, 'BBB': 1, 'CCC': 2, 'DDD': 3, 'EEE': 4, 'FFF': 5, 'GGG': 6}

13 model features: ['date', 'weight', 'num_of_prev_attempts', 'studied_credits', 'module_presentation_length', 'date_registration', 'prior_avg_score', 'avg_weekly_clicks', 'clicks_trend', 'active_weeks_ratio', 'has_vle_activity', 'assessment_type_enc', 'code_module_enc']


In [17]:
clean_df = df_model[model_feature_cols + ["Priority_Label"]]
# ~10 significant digits: the exact number format of the committed file,
# so re-running this notebook reproduces it byte-for-byte instead of a whole-file git diff.
def ten_digits(x):
    return f"{x:.10g}" if abs(x) >= 1 or x == 0 else f"{round(x, 9):.10g}"


clean_df.to_csv(CLEAN_CSV_PATH, index=False, float_format=ten_digits)
print(f"Saved {clean_df.shape[0]:,} rows x {clean_df.shape[1]} columns -> {CLEAN_CSV_PATH}")
clean_df.head()

Saved 170,874 rows x 14 columns -> C:\Users\Chethana Methsithum\Desktop\Smart-Uni-Guide\backend\ml_scripts\study-planner\outputs\oulad_task_level_leakage_free.csv


,date,weight,num_of_prev_attempts,studied_credits,module_presentation_length,date_registration,prior_avg_score,avg_weekly_clicks,clicks_trend,active_weeks_ratio,has_vle_activity,assessment_type_enc,code_module_enc,Priority_Label
0,19,10.0,0,60,269,-52.0,68.59868,71.564103,-71.394737,1.0,1,2,0,Medium
1,54,20.0,0,60,269,-52.0,60.00000,71.564103,-71.394737,1.0,1,2,0,High
2,117,20.0,0,60,269,-52.0,54.00000,71.564103,-71.394737,1.0,1,2,0,Medium
3,166,20.0,0,60,269,-52.0,57.00000,71.564103,-71.394737,1.0,1,2,0,Medium
4,215,30.0,0,60,269,-52.0,58.00000,71.564103,-71.394737,1.0,1,2,0,Medium


> **Note:** `date_registration` still has a handful of `NaN`s here on purpose. XGBoost handles missing values natively; the training notebook median-imputes them only for the models that can't (Logistic Regression, SVM, KNN, …).

## 11. Exploratory data analysis (final table)

In [18]:
summary = clean_df.describe().T
summary["missing"] = clean_df[model_feature_cols].isna().sum()
summary.round(2)

,count,mean,std,min,25%,50%,75%,max,missing
date,170874.0,130.63,78.03,12.00,54.00,129.00,214.00,261.00,0
weight,170874.0,11.28,13.98,0.00,0.00,9.00,18.00,100.00,0
num_of_prev_attempts,170874.0,0.15,0.45,0.00,0.00,0.00,0.00,6.00,0
studied_credits,170874.0,76.81,37.36,30.00,60.00,60.00,90.00,630.00,0
module_presentation_length,170874.0,255.17,13.60,234.00,240.00,261.00,268.00,269.00,0
date_registration,170867.0,-65.85,46.97,-311.00,-94.00,-53.00,-29.00,167.00,7
prior_avg_score,170874.0,76.45,13.16,0.00,70.00,78.50,85.00,100.00,0
avg_weekly_clicks,170874.0,63.01,52.52,0.00,26.21,46.03,85.27,652.68,0
clicks_trend,170874.0,-31.65,113.53,-630.82,-72.26,-31.63,-11.20,2713.80,0
active_weeks_ratio,170874.0,1.00,0.03,0.00,1.00,1.00,1.00,1.00,0


In [19]:
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
for ax, col in zip(axes.flat, model_feature_cols):
    sns.histplot(clean_df[col].dropna(), bins=40, ax=ax, color="#6e56cf")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")
for ax in list(axes.flat)[len(model_feature_cols):]:
    ax.set_visible(False)
fig.suptitle("Feature distributions", fontsize=14)
plt.tight_layout()
save_fig("prep_feature_distributions")

Saved figure: ml_scripts\study-planner\outputs\figures\prep_feature_distributions.png


![Feature distributions](outputs/figures/prep_feature_distributions.png)

In [20]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
order = ["Low", "Medium", "High"]
palette = {"High": "#e5484d", "Medium": "#f5a524", "Low": "#30a46c"}
for ax, col in zip(axes, ["weight", "prior_avg_score", "avg_weekly_clicks"]):
    sns.boxplot(data=clean_df, x="Priority_Label", y=col, order=order, palette=palette, ax=ax, showfliers=False)
    ax.set_title(f"{col} by Priority_Label")
plt.tight_layout()
save_fig("prep_features_by_label")

Saved figure: ml_scripts\study-planner\outputs\figures\prep_features_by_label.png


![Features by Priority_Label](outputs/figures/prep_features_by_label.png)

In [21]:
corr = clean_df[model_feature_cols].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Feature correlation matrix")
plt.tight_layout()
save_fig("prep_correlation")

Saved figure: ml_scripts\study-planner\outputs\figures\prep_correlation.png


![Feature correlation matrix](outputs/figures/prep_correlation.png)

## Summary

* Raw OULAD tables were cleaned (`?` placeholders removed or coerced), joined into one **task-level** table, and enriched with **chunk-processed engagement features** from 10.6M click rows.
* `prior_avg_score` uses only **earlier** assessments, so it never contains the current task's outcome.
* `Priority_Label` is built from the **real outcome** (score, lateness, weight). The outcome columns were then removed from the features, which fixes the original 99%-accuracy leakage bug.
* The output `oulad_task_level_leakage_free.csv` (13 features + label) feeds **`model_training.ipynb`**.